# B실험 300장: PaddleOCR(기본 / 검출 기준 0.0) × 전처리 11개

**시작 전에:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 를 선택하세요.

위에서부터 순서대로 실행해요. 바꿀 곳은 **3번 셀의 zip 경로**뿐이에요.
A실험(남예원)과 **같은 사진·같은 분할·같은 채점 방식**으로 PaddleOCR 두 설정을 돌립니다.
- `paddleocr`: PaddleOCR 기본 검출 기준 / `paddleocr_t0`: 검출 기준 0.0 (팀원 B 실험 설정)

| 순서 | 하는 일 | 걸리는 시간(예상) |
|---|---|---|
| 1~2 | 드라이브 연결, 코드 받기, PaddleOCR 설치 | 3~5분 |
| 3~4 | zip 풀기, 채점표·검색 DB 만들기 (같은 시드라 A실험과 같은 분할) | 1분 |
| 5 | **시험 운전**: 5장만 돌려서 PaddleOCR이 켜지는지 확인 | 3~8분 |
| 6 | **1단계**: 개발용(51장) × 2설정 × 전처리 11개 | 2~4시간 |
| 7 | 1단계 채점 → **설정마다** 가장 좋은 전처리 1개 (22개 순위표) | 1분 |
| 8 | **2단계**: 평가용(237장) × 고른 2개 (한 번만!) | 1~3시간 |
| 9 | 최종 채점 + 혼동행렬 | 1분 |
| 10 | A실험 결과와 한 표로 보기 | 1초 |

결과는 모두 드라이브 `exp300_results_paddle` 폴더에 바로 저장돼요. **끊기면 같은 셀을 다시 실행**하면 이어서 해요.

**전처리 11개 = P0~P3(A실험과 같음) + P4 + 추가 6개(P5~P10)**
`P0` 회색조(grayscale) · `P1` 조명평탄화 · `P2` 반사광제거+평탄화 · `P3` CLAHE · `P4` 2배확대+CLAHE · `P5` adaptive_threshold · `P6` blackhat · `P7` closing · `P8` global_otsu · `P9` localized_otsu · `P10` tophat

> ⚠️ **8번(평가용) 점수를 보고 전처리를 다시 고르면 안 돼요.** 고르는 건 7번(개발용)에서만 합니다.


In [ ]:
# 1) 드라이브 연결 + 코드 최신으로 받기
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.exists('/content/dl_deep_imgps'):
    %cd /content/dl_deep_imgps
    !git pull -q
else:
    %cd /content
    !git clone -q https://github.com/yewon51019/dl_deep_imgps
    %cd /content/dl_deep_imgps
!nvidia-smi -L

In [ ]:
# 2) PaddleOCR 설치  (설치 후 import 오류가 나면: 런타임 → 세션 다시 시작 → 1번부터 다시)
!pip -q install paddlepaddle paddleocr
import paddleocr
print('PaddleOCR 버전:', getattr(paddleocr, '__version__', '?'))

In [ ]:
# 3) ★ zip 경로 — A실험과 '같은 zip 파일' 두 개를 써야 같은 사진·같은 분할이 돼요
ZIP_FILES = [
    '/content/drive/MyDrive/Colab Notebooks/pill_images_test/약제각인인식_테스트셋.zip',
    '/content/drive/MyDrive/Colab Notebooks/pill_images_test/약제각인인식_테스트셋_2.zip',
]
RESULT = '/content/drive/MyDrive/exp300_results_paddle'   # 결과 저장 폴더 (A실험 폴더와 섞이지 않게 따로)
os.makedirs(RESULT, exist_ok=True)
for z in ZIP_FILES:
    print('있음 ✓' if os.path.exists(z) else '없음 ✗ (경로 확인!)', z)

In [ ]:
# 4) zip 풀기 + 채점표·검색 DB 만들기 (시드가 같아서 A실험과 같은 분할)
import subprocess, csv, sys
# ※ 예전에 잘못된 zip을 풀어둔 적이 있으면 먼저:  !rm -rf /content/data1 /content/data2
ROOTS = []
for i, z in enumerate(ZIP_FILES, 1):
    out = f'/content/data{i}'
    if not os.path.exists(out):
        subprocess.run(['unzip', '-q', '-o', z, '-d', out], check=True)
    for cur, subdirs, _ in os.walk(out):
        if 'images' in subdirs and 'labels' in subdirs:
            ROOTS.append(cur)
print('찾은 폴더:', ROOTS)
MAN, DB = f'{RESULT}/manifest.csv', f'{RESULT}/search_db.csv'
!python exp300/make_manifest.py --roots {' '.join(repr(r) for r in ROOTS)} --out "{MAN}" --dev-ratio 0.2
!python exp300/make_search_db.py --label-roots {' '.join(repr(r + '/labels') for r in ROOTS)} --out "{DB}"

# ✔ A실험과 같은 분할인지 확인 (기대값: 개발 51장 / 평가 237장 / 채점 제외 12장 / 약 27종)
sys.path.insert(0, '/content/dl_deep_imgps/exp300')
from run_exp300 import norm
rows = list(csv.DictReader(open(MAN, encoding='utf-8-sig')))
dev  = sum(1 for r in rows if r['split'] == 'dev'  and norm(r['truth']))
test = sum(1 for r in rows if r['split'] == 'test' and norm(r['truth']))
skip = sum(1 for r in rows if not norm(r['truth']))
drugs = len({r['drug_N'] for r in rows})
print(f'개발 {dev}장 / 평가 {test}장 / 채점 제외 {skip}장 / 약 {drugs}종')
print('✅ A실험과 같은 분할' if (dev, test, skip, drugs) == (51, 237, 12, 27) else '⚠️ A실험과 달라요! zip 파일이 같은지 확인하세요')

In [ ]:
# 5) 시험 운전: 개발용 5장 × 2설정 × 전처리 11개. '[건너뜀]'이 안 나오고 읽은 글자가 찍히는지 확인!
#    (처음엔 PaddleOCR 모델 파일을 내려받느라 느려요)
!python exp300/run_exp300_paddle.py --manifest "{MAN}" --db "{DB}" --split dev --limit 5 --out /content/pred_smoke_paddle.csv
!python exp300/evaluate_exp300.py --pred /content/pred_smoke_paddle.csv --out-dir /content/eval_smoke_paddle --top 3

In [ ]:
# 6) 1단계: 개발용 전부 × 2설정 × 전처리 11개  (끊기면 이 셀만 다시 실행 → 이어서 해요)
!python exp300/run_exp300_paddle.py --manifest "{MAN}" --db "{DB}" --split dev --out "{RESULT}/pred_dev_paddle.csv"

In [ ]:
# 7) 1단계 채점: 22개 순위표 → '설정마다' 가장 좋은 전처리 1개를 고름
!python exp300/evaluate_exp300.py --pred "{RESULT}/pred_dev_paddle.csv" --out-dir "{RESULT}/eval_dev_paddle" --top 3
rows = list(csv.DictReader(open(f'{RESULT}/eval_dev_paddle/summary.csv', encoding='utf-8-sig')))   # 순위 순서
best = {}
for r in rows:
    if float(r['edit_F1']) <= 0:                   # 점수가 0인 조합(전부 못 읽음)은 '고른' 것이 아니므로 제외
        continue
    best.setdefault(r['model'], r['combo'])        # 설정마다 가장 먼저 나온(=가장 좋은) 조합
TOP = ','.join(best.values())
print('\n2단계에 쓸 조합 (설정마다 1개):', TOP)
skipped = {r['model'] for r in rows} - set(best)
if skipped:
    print('⚠️ 점수가 전부 0이라 제외한 설정:', skipped, '→ 7-1번 셀로 원인을 확인하세요')

In [ ]:
# 7-1) (선택) 기본 검출 기준이 왜 전부 빈 글자인지 확인 — 개발용 4장, 몇 분
!python exp300/diag_paddle.py --manifest "{MAN}" --n 4

In [ ]:
# 8) 2단계: 평가용 × 고른 2개 조합 (설정마다 1개)  (끊기면 다시 실행)
#    ※ 여기서 나온 점수를 보고 전처리를 다시 고르면 안 돼요 (그러면 평가용이 개발용이 돼버림)
#    ※ 런타임이 끊겨서 TOP 이 없으면: 1~4번 셀 → 7번 셀 다시 실행 후 이 셀
!python exp300/run_exp300_paddle.py --manifest "{MAN}" --db "{DB}" --split test --combos "{TOP}" --out "{RESULT}/pred_test_paddle.csv"

In [ ]:
# 9) 최종 채점 + 혼동행렬
!python exp300/evaluate_exp300.py --pred "{RESULT}/pred_test_paddle.csv" --out-dir "{RESULT}/eval_test_paddle" --top 2 --cm all
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{RESULT}/eval_test_paddle/cm_*.png')):
    print(p); display(Image(p))

In [ ]:
# 10) A실험 결과와 한 표로 보기 (평가용 237장, edit 매칭 기준, 단위 %)
A = [  # A실험 2단계 결과 (2026-10-10)
    ('parseq:P0',    66.7, 81.4, 70.9, 72.9, 38.8),
    ('trocr:P3',     53.6, 70.7, 63.0, 64.7, 31.6),
    ('easyocr:P0',   39.2, 69.0, 48.6, 54.3, 15.2),
    ('doctr:P4',     25.7, 46.2, 31.9, 35.6,  9.3),
    ('tesseract:P1',  2.5, 13.6,  3.3,  4.8,  0.8),
]
S = list(csv.DictReader(open(f'{RESULT}/eval_test_paddle/summary.csv', encoding='utf-8-sig')))
B = [(s['combo'] + ' ★', *(round(float(s[k]) * 100, 1) for k in ('edit_acc', 'edit_P', 'edit_R', 'edit_F1', 'imp_exact'))) for s in S]
print(f"{'조합':<18} {'정확도':>6} {'정밀도':>6} {'재현율':>6} {'F1':>6} {'각인일치':>8}   (평가용 n={S[0]['n']})")
for r in sorted(A + B, key=lambda r: -r[4]):
    print(f'{r[0]:<18} {r[1]:6.1f} {r[2]:6.1f} {r[3]:6.1f} {r[4]:6.1f} {r[5]:8.1f}')


In [ ]:
# 11) (선택) 글자 검출 박스 그림: 300장 중 30장(회전각 6종×5장, 카메라 60/90° 반반)에 CRAFT / Paddle 기본 / Paddle 0.0 박스를 그려서 저장
#     본 실험 점수는 건드리지 않아요. 몇 분 걸려요. 결과: 드라이브 exp300_results_paddle/viz_boxes/ (사진별 PNG, sheet_*.png, boxes.csv)
!python exp300/viz_boxes_exp300.py --manifest "{MAN}" --out "{RESULT}/viz_boxes" --n 30 --seed 42
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{RESULT}/viz_boxes/sheet_*.png'))[:2]:
    display(Image(p))